# Tugas Praktikum Pertemuan 9: Masalah Sinkronisasi II — Classical Problems

**Nama:** Rasya Ramadhani  
**NIM:** 2411511006  
---


## 1️⃣ Soal 1: Dining Philosophers dengan $N$ Berbeda ($N = 3, 7, 10$)

### Penjelasan Pendekatan
Pada solusi **Arbitrator (Waiter)**, digunakan `threading.Semaphore(N - 1)` untuk membatasi agar maksimal $N - 1$ filsuf yang dapat masuk ke area pengambilan garpu secara bersamaan. Hal ini memutus syarat *circular wait* karena selalu ada minimal satu filsuf yang tidak memegang garpu sama sekali, sehingga tetangganya dapat memperoleh kedua garpu.

Di bawah ini kita menjalankan simulasi dengan jumlah putaran makan per filsuf tetap (`putaran = 20`), lalu mengukur total waktu eksekusi dan menghitung **throughput** ($\text{Total Makan} / \text{Detik}$) untuk $N = 3$, $N = 7$, dan $N = 10$.


In [1]:
import threading
import time

def jalankan_dining_arbitrator(N, putaran=20):
    forks = [threading.Lock() for _ in range(N)]
    kursi_meja = threading.Semaphore(N - 1)
    selesai_makan = [0] * N

    def filsuf(i):
        kiri = forks[i]
        kanan = forks[(i + 1) % N]
        for _ in range(putaran):
            with kursi_meja:
                kiri.acquire()
                kanan.acquire()
                selesai_makan[i] += 1
                time.sleep(0.02)  # simulasi waktu makan
                kanan.release()
                kiri.release()
            time.sleep(0.01)      # simulasi waktu berpikir

    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]
    t_mulai = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    durasi = time.perf_counter() - t_mulai

    total_makan = sum(selesai_makan)
    throughput = total_makan / durasi
    return total_makan, durasi, throughput, selesai_makan

print(f"{'N':<5} | {'Total Makan':<12} | {'Waktu (detik)':<15} | {'Throughput (makan/dtk)':<22}")
print("-" * 62)
for n_filsuf in [3, 7, 10]:
    total, durasi, tp, detail = jalankan_dining_arbitrator(n_filsuf, putaran=20)
    print(f"{n_filsuf:<5} | {total:<12} | {durasi:<15.4f} | {tp:<22.2f}")

N     | Total Makan  | Waktu (detik)   | Throughput (makan/dtk)
--------------------------------------------------------------
3     | 60           | 1.2576          | 47.71                 
7     | 140          | 0.9983          | 140.24                
10    | 200          | 0.8590          | 232.84                


### Analisis Hasil Soal 1

**Apakah throughput naik linear terhadap $N$?**  
**Tidak.** Throughput memang meningkat seiring bertambahnya $N$, tetapi **tidak naik secara linear sempurna** ($TP(10) \neq \frac{10}{3} TP(3)$). Penyebab utamanya adalah:

1. **Batas Maksimum Konkurensi Fisik Meja ($\lfloor N/2 \rfloor$):** Setiap filsuf membutuhkan **2 garpu** yang bersebelahan untuk makan. Dengan $N$ garpu di meja, jumlah maksimal filsuf yang benar-benar bisa makan secara bersamaan (paralel) hanyalah $\lfloor N/2 \rfloor$:
   - Untuk $N = 3$: maksimal hanya **1** filsuf yang bisa makan pada saat bersamaan.
   - Untuk $N = 7$: maksimal **3** filsuf yang bisa makan bersamaan.
   - Untuk $N = 10$: maksimal **5** filsuf yang bisa makan bersamaan.
2. **Contention pada Garpu & Arbitrator:** Walaupun arbitrator mengizinkan $N-1$ filsuf masuk, sisanya tetap harus menunggu *lock* garpu tetangganya yang sedang dipakai. Selain itu, bertambahnya jumlah thread juga menambah sedikit *overhead* penjadwalan (*context switching*) di sistem operasi dan Python GIL.


## 2️⃣ Soal 2: Readers-Writers dengan Prioritas Writer (*Writer-Preference*)

### Penjelasan Mekanisme
Untuk membuat solusi yang **memprioritaskan writer**, kita menggunakan dua penghitung (`read_count` dan `write_count`) beserta dua *lock* pintu utama:
- `resource_lock` (`write_lock`): Mengatur akses eksklusif ke data bersama.
- `read_try`: Pintu masuk bagi reader. Begitu **writer pertama** datang (`write_count == 1`), writer akan mengunci `read_try`. Akibatnya, **tidak ada reader baru** yang bisa melewati `read_try` selama masih ada writer yang menunggu atau sedang menulis.
- Reader yang **sudah berada di dalam** (sudah melewati `read_try` sebelum writer datang) tetap dibiarkan menyelesaikan bacaannya, dan reader terakhir yang keluar (`read_count == 0`) akan melepaskan `resource_lock` agar writer bisa langsung masuk.

Di bawah ini kita menguji skenario di mana **writer datang terus-menerus** sementara sebuah **reader baru** mencoba membaca.


In [2]:
import threading
import time

read_count = 0
write_count = 0

mutex_read = threading.Lock()
mutex_write = threading.Lock()
read_try = threading.Lock()       # Gerbang untuk memblokir reader baru saat ada writer menunggu
resource_lock = threading.Lock()  # Akses eksklusif ke data bersama

berhenti = threading.Event()
jumlah_tulis = [0]
reader_berhasil = threading.Event()

def writer_terus_menerus():
    global write_count
    while not berhenti.is_set():
        # Bagian Masuk Writer
        with mutex_write:
            write_count += 1
            if write_count == 1:
                read_try.acquire()  # Writer PERTAMA menutup gerbang bagi reader baru

        # Critical Section (Menulis)
        with resource_lock:
            jumlah_tulis[0] += 1
            time.sleep(0.005)

        # Bagian Keluar Writer
        with mutex_write:
            write_count -= 1
            if write_count == 0:
                read_try.release()  # Writer TERAKHIR membuka kembali gerbang reader

def satu_reader_baru():
    global read_count
    t0 = time.perf_counter()
    
    # Reader harus melewati gerbang read_try terlebih dahulu
    with read_try:
        with mutex_read:
            read_count += 1
            if read_count == 1:
                resource_lock.acquire()

    # Critical Section (Membaca)
    reader_berhasil.set()
    waktu_tunggu = time.perf_counter() - t0
    print(f"Reader baru akhirnya berhasil membaca setelah menunggu {waktu_tunggu:.2f} detik")

    # Bagian Keluar Reader
    with mutex_read:
        read_count -= 1
        if read_count == 0:
            resource_lock.release()

# Jalankan 4 writer terus-menerus lebih dulu
writers = [threading.Thread(target=writer_terus_menerus) for _ in range(4)]
for w in writers:
    w.start()

time.sleep(0.1)  # Biarkan writer aktif mengantre

# Coba masukkan 1 reader baru saat writer terus berdatangan
reader = threading.Thread(target=satu_reader_baru)
reader.start()
reader.join(timeout=2.0)

if not reader_berhasil.is_set():
    print("Reader baru BELUM juga mendapat giliran setelah 2.00 detik -- READER STARVATION TERBUKTI!")

berhenti.set()
for w in writers:
    w.join()
reader.join()
print(f"Total operasi tulis yang terjadi selama reader menunggu: {jumlah_tulis[0]}")

Reader baru BELUM juga mendapat giliran setelah 2.00 detik -- READER STARVATION TERBUKTI!
Reader baru akhirnya berhasil membaca setelah menunggu 2.02 detik
Total operasi tulis yang terjadi selama reader menunggu: 394


### Analisis Hasil Soal 2

Hasil pengujian di atas membuktikan bahwa **reader baru mengalami starvation** (*Reader Starvation*). Ketika sekelompok writer terus berdatangan tanpa jeda:
1. Nilai `write_count` selalu $\ge 1$, sehingga `read_try` tetap terkunci sepanjang waktu.
2. Thread `satu_reader_baru` tertahan di `with read_try:` selama penuh 2 detik hingga arus writer dihentikan (`berhenti.set()`).
3. Hal ini menunjukkan sifat simetris pada *Readers-Writers Problem*: memprioritaskan reader menyebabkan *writer starvation*, sedangkan memprioritaskan writer secara mutlak menyebabkan *reader starvation*. Untuk mencegah keduanya, diperlukan mekanisme antrean FIFO bersama (*fair solution*).


## 3️⃣ Soal 3: Sleeping Barber dengan Banyak Tukang Cukur (2 Tukang Cukur)

### Penjelasan Desain Sinkronisasi
Ketika terdapat **2 tukang cukur** (`JUMLAH_BARBER = 2`) yang berbagi satu ruang tunggu dengan kapasitas terbatas (`KURSI_TUNGGU = 3`), kita tidak cukup hanya menggunakan satu semaphore siap-layani yang anonim karena pelanggan perlu mengetahui **tukang cukur mana** yang memanggilnya.

Solusi yang digunakan di sini memadukan `Semaphore` dan `queue.Queue`:
1. `pelanggan_menunggu = threading.Semaphore(0)`: Menghitung pelanggan yang sudah duduk di ruang tunggu dan membangunkan tukang cukur yang sedang tidur.
2. `barber_siap_queue = queue.Queue()`: Menyimpan ID tukang cukur yang sedang kosong/siap (`Barber-0` atau `Barber-1`), sekaligus menyediakan `Semaphore` pribadi milik tukang cukur tersebut untuk mengoordinasikan momen duduk hingga selesai cukur.
3. `mutex = threading.Lock()`: Melindungi pengecekan dan pembaruan variabel `jumlah_menunggu` agar kapasitas kursi tunggu (`3`) tidak terlampaui.


In [3]:
import threading
import queue
import time
import random

KURSI_TUNGGU = 3
JUMLAH_BARBER = 2
TOTAL_PELANGGAN = 10

pelanggan_menunggu = threading.Semaphore(0)
barber_siap_queue = queue.Queue()  # Antrean tukang cukur yang siap: berisi (id_barber, sem_mulai, sem_selesai)
mutex = threading.Lock()
jumlah_menunggu = [0]

log_lock = threading.Lock()
def catat(msg):
    with log_lock:
        print(msg)

def tukang_cukur(id_barber):
    sem_mulai = threading.Semaphore(0)
    sem_selesai = threading.Semaphore(0)
    while True:
        catat(f"[Barber-{id_barber}] tidur / menunggu pelanggan...")
        pelanggan_menunggu.acquire()  # Tidur sampai ada pelanggan di ruang tunggu

        with mutex:
            jumlah_menunggu[0] -= 1
            sisa_tunggu = jumlah_menunggu[0]

        # Daftarkan diri ke antrean barber siap agar pelanggan tahu siapa yang memanggil
        barber_siap_queue.put((id_barber, sem_mulai, sem_selesai))

        # Tunggu pelanggan duduk di kursi cukur Barber-{id_barber}
        sem_mulai.acquire()
        catat(f"[Barber-{id_barber}] mulai mencukur pelanggan (sisa di ruang tunggu: {sisa_tunggu}/{KURSI_TUNGGU})")
        time.sleep(random.uniform(0.15, 0.22))  # Simulasi durasi potong rambut
        catat(f"[Barber-{id_barber}] selesai mencukur!")
        sem_selesai.release()

def pelanggan(id_pelanggan):
    with mutex:
        if jumlah_menunggu[0] < KURSI_TUNGGU:
            jumlah_menunggu[0] += 1
            catat(f"[Pelanggan-{id_pelanggan}] duduk di ruang tunggu ({jumlah_menunggu[0]}/{KURSI_TUNGGU})")
            masuk = True
        else:
            catat(f"[Pelanggan-{id_pelanggan}] ruang tunggu PENUH ({KURSI_TUNGGU}/{KURSI_TUNGGU}), pergi!")
            masuk = False

    if masuk:
        pelanggan_menunggu.release()  # Bangunkan salah satu tukang cukur
        id_barber, sem_mulai, sem_selesai = barber_siap_queue.get()  # Ambil giliran tukang cukur yang tersedia
        catat(f"  -> [Pelanggan-{id_pelanggan}] dilayani oleh [Barber-{id_barber}]")
        sem_mulai.release()    # Beri sinyal ke barber bahwa pelanggan sudah duduk
        sem_selesai.acquire()  # Tunggu sampai proses cukur selesai

# Jalankan 2 thread tukang cukur
for b in range(JUMLAH_BARBER):
    threading.Thread(target=tukang_cukur, args=(b,), daemon=True).start()

time.sleep(0.05)

# Simulasikan kedatangan 10 pelanggan secara acak
random.seed(42)
pelanggan_threads = []
for p in range(TOTAL_PELANGGAN):
    time.sleep(random.uniform(0.02, 0.08))
    t = threading.Thread(target=pelanggan, args=(p,))
    t.start()
    pelanggan_threads.append(t)

for t in pelanggan_threads:
    t.join()

time.sleep(0.05)
print("\nSimulasi selesai: semua pelanggan telah dilayani oleh salah satu barber atau pergi karena ruang tunggu penuh.")

[Barber-0] tidur / menunggu pelanggan...
[Barber-1] tidur / menunggu pelanggan...
[Pelanggan-0] duduk di ruang tunggu (1/3)
  -> [Pelanggan-0] dilayani oleh [Barber-0]
[Barber-0] mulai mencukur pelanggan (sisa di ruang tunggu: 0/3)
[Pelanggan-1] duduk di ruang tunggu (1/3)
  -> [Pelanggan-1] dilayani oleh [Barber-1]
[Barber-1] mulai mencukur pelanggan (sisa di ruang tunggu: 0/3)
[Pelanggan-2] duduk di ruang tunggu (1/3)
[Pelanggan-3] duduk di ruang tunggu (2/3)
[Barber-0] selesai mencukur!
[Barber-0] tidur / menunggu pelanggan...
  -> [Pelanggan-2] dilayani oleh [Barber-0]
[Barber-0] mulai mencukur pelanggan (sisa di ruang tunggu: 1/3)
[Barber-1] selesai mencukur!
[Barber-1] tidur / menunggu pelanggan...
  -> [Pelanggan-3] dilayani oleh [Barber-1]
[Barber-1] mulai mencukur pelanggan (sisa di ruang tunggu: 0/3)
[Pelanggan-4] duduk di ruang tunggu (1/3)
[Pelanggan-5] duduk di ruang tunggu (2/3)
[Pelanggan-6] duduk di ruang tunggu (3/3)
[Barber-0] selesai mencukur!
[Barber-0] tidur / menu

### Analisis Hasil Soal 3

Dengan menggunakan `barber_siap_queue` (`queue.Queue`), kedua tukang cukur (`Barber-0` dan `Barber-1`) dapat bekerja secara paralel melayani dua pelanggan berbeda dalam waktu bersamaan tanpa tertukar sinyal:
- Saat salah satu tukang cukur bangun dan mengambil satu pelanggan dari ruang tunggu, ia memasukkan identitas dan pasangan semaphore pribadinya `(id_barber, sem_mulai, sem_selesai)` ke `barber_siap_queue`.
- Pelanggan yang mendapat giliran memanggil `barber_siap_queue.get()` secara atomik dan *thread-safe*, sehingga terikat secara eksklusif ke tukang cukur tersebut sampai proses cukur selesai (`sem_selesai.acquire()`).
- Apabila kedua tukang cukur sedang sibuk dan ketiga kursi tunggu terisi penuh (`3/3`), pelanggan yang baru datang langsung ditolak dan pergi.


## 4️⃣ Soal 4: Refleksi Singkat (Maks. 150 Kata)

Dari kelima *classical synchronization problems*, **The Producer-Consumer Problem** dan **The Sleeping Barber Problem** adalah pola yang paling sering ditemui dalam pengembangan perangkat lunak nyata, khususnya pada arsitektur **web server dan API gateway**.

Sebagai contoh konkret, pada layanan backend berbasis Python (seperti FastAPI atau Gunicorn dengan *worker pool*), setiap permintaan HTTP dari klien berperan sebagai **pelanggan**, *thread/worker pool* berperan sebagai **tukang cukur**, dan *request queue* dengan kapasitas maksimum berperan sebagai **kursi tunggu**. Ketika beban trafik normal, *worker* memproses permintaan secara paralel layaknya pola *Producer-Consumer*. Namun, saat terjadi lonjakan trafik (*traffic spike*) hingga antrean permintaan penuh, server menerapkan prinsip **Sleeping Barber** dengan langsung menolak permintaan baru (mengembalikan status HTTP `503 Service Unavailable` atau `429 Too Many Requests`) alih-alih membiarkan memori habis akibat antrean tanpa batas.
